# UmarTransit v1.0: Retrain + Benchmark Evaluation

Retrain on expanded dataset (3,501 pairs vs original 3,306) with new categories:
journey planning, GTFS validation, transit operations, expanded GTFS knowledge.

Then run the 193-question benchmark to compare v0.1 vs v1.0.

> **Runtime:** Select **T4 GPU**. Total time ~45-60 min (training ~30 min + benchmark ~15 min).

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install rouge-score nltk

## 2. Configuration

In [ ]:
# ── Model ──
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MODEL_NAME = "UmarTransit-1B"
MAX_SEQ_LENGTH = 1024

# ── LoRA ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ── Training ──
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
SEED = 42

# ── Output ──
OUTPUT_DIR = "./umartransit-v1-output"
HF_REPO_ID = "umarfarookm/UmarTransit-1B"

# ── Dataset URLs ──
GITHUB_RAW = "https://raw.githubusercontent.com/umarfarookm/transit-foundation-model/main"
TRAIN_URL = f"{GITHUB_RAW}/datasets/synthetic/train.jsonl"
TEST_URL = f"{GITHUB_RAW}/datasets/synthetic/test.jsonl"
BENCHMARK_URL = f"{GITHUB_RAW}/evaluation/benchmark.json"
METRICS_URL = f"{GITHUB_RAW}/evaluation/metrics.py"

# ── System prompt ──
SYSTEM_PROMPT = (
    "You are UmarTransit-1B, a specialized AI assistant for public transit systems "
    "and GTFS (General Transit Feed Specification) data. You provide accurate, "
    "detailed answers about transit routes, stops, schedules, transfers, and GTFS concepts."
)

# ── Inference ──
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1
TOP_P = 0.9

## 3. Load Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
)
print(f"Model loaded: {BASE_MODEL}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 4. Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

## 5. Load Expanded Dataset (3,501 pairs)

In [ ]:
import json
import requests
from datasets import Dataset

def load_jsonl_url(url):
    r = requests.get(url)
    r.raise_for_status()
    return [json.loads(line) for line in r.text.strip().split("\n") if line.strip()]

try:
    print("Loading from GitHub...")
    train_data = load_jsonl_url(TRAIN_URL)
    test_data = load_jsonl_url(TEST_URL)
except Exception as e:
    print(f"GitHub failed: {e}")
    print("Upload train.jsonl and test.jsonl manually.")
    train_data = [json.loads(l) for l in open("train.jsonl")]
    test_data = [json.loads(l) for l in open("test.jsonl")]

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

from collections import Counter
cats = Counter(d['category'] for d in train_data + test_data)
print("\nCategories:")
for c, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {c}: {n}")

In [ ]:
def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_dataset = Dataset.from_list(train_data).map(format_chat)
print(f"Formatted {len(train_dataset)} training examples")

## 6. Train (~30 min on T4)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        logging_steps=LOGGING_STEPS,
        save_strategy="no",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        seed=SEED,
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False,
    ),
)

print(f"Steps per epoch: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")
print(f"Total steps: ~{NUM_EPOCHS * len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")
print("\nStarting training...")

stats = trainer.train()

print(f"\nDone! Loss: {stats.training_loss:.4f} | Steps: {stats.global_step} | Time: {stats.metrics['train_runtime']:.0f}s")

## 7. Quick Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, do_sample=True)
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

# Test on new categories
test_qs = [
    "What is headway in public transit?",
    "Is a GTFS feed valid without calendar.txt or calendar_dates.txt?",
    "What factors should a journey planner consider when calculating the fastest route?",
    "What does route_type 3 mean in GTFS?",
    "How many routes does the Chicago Transit Authority operate?",
]

for q in test_qs:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)}")
    print("-" * 60)

## 8. Push v1.0 to Hugging Face

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
)
print(f"\nv1.0 pushed to: https://huggingface.co/{HF_REPO_ID}")

## 9. Benchmark: v1.0 vs v0.1

Run 193 benchmark questions on both the old model (v0.1) and the newly trained v1.0.

First, free GPU memory and reload v1.0 as a standard Transformers model.

In [ ]:
# Free training model from memory
del model, tokenizer, trainer
torch.cuda.empty_cache()
import gc
gc.collect()
print(f"GPU memory freed: {torch.cuda.memory_allocated() / 1024**3:.1f} GB used")

In [ ]:
# Download benchmark and metrics
for fname, url in [("benchmark.json", BENCHMARK_URL), ("metrics.py", METRICS_URL)]:
    print(f"Downloading {fname}...")
    r = requests.get(url)
    if r.status_code == 200:
        with open(fname, "w") as f:
            f.write(r.text)
        print(f"  OK ({len(r.text):,} bytes)")
    else:
        print(f"  FAILED ({r.status_code}) — upload manually")

with open("benchmark.json") as f:
    benchmark = json.load(f)
questions = benchmark["questions"]
print(f"\nBenchmark: {len(questions)} questions")

from metrics import score_response, score_benchmark, print_report, report_to_dict

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTokenizer
from tqdm import tqdm
import time

def load_hf_model(model_id):
    print(f"Loading {model_id}...")
    tok = HFTokenizer.from_pretrained(model_id)
    mdl = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
    mdl.eval()
    print(f"  GPU: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
    return mdl, tok

def generate(mdl, tok, question, sys_prompt):
    messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": question}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = mdl.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, do_sample=True)
    return tok.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

def run_benchmark(mdl, tok, questions, model_id, sys_prompt):
    results = []
    for q in tqdm(questions, desc=model_id.split('/')[-1]):
        t0 = time.time()
        gen = generate(mdl, tok, q["question"], sys_prompt)
        elapsed = time.time() - t0
        s = score_response(generated=gen, expected=q["expected_answer"], scoring_criteria=q["scoring_criteria"])
        results.append({
            "id": q["id"], "question": q["question"], "expected_answer": q["expected_answer"],
            "generated": gen, "category": q["category"], "difficulty": q["difficulty"],
            "scoring_criteria": q["scoring_criteria"],
            "rouge_l": round(s.rouge_l, 4), "keyword_match": round(s.keyword_match, 4),
            "criteria_match": round(s.criteria_match, 4), "combined": round(s.combined, 4),
            "time_seconds": round(elapsed, 2),
        })
    return results

### 9a. Evaluate v1.0 (~10-15 min)

In [ ]:
v1_model, v1_tok = load_hf_model(HF_REPO_ID)
v1_results = run_benchmark(v1_model, v1_tok, questions, HF_REPO_ID, SYSTEM_PROMPT)
v1_report = score_benchmark(v1_results, model_id=f"{HF_REPO_ID} (v1.0)")
print_report(v1_report)
v1_dict = report_to_dict(v1_report)

del v1_model, v1_tok
torch.cuda.empty_cache()

### 9b. Load v0.1 results for comparison

We use the v0.1 results you already have from the previous benchmark run.

In [ ]:
# v0.1 results from previous benchmark (hardcoded from comparison.json)
v01_dict = {
    "model_id": "umarfarookm/UmarTransit-1B (v0.1)",
    "total_questions": 193,
    "overall": {
        "rouge_l": 0.3751,
        "keyword_match": 0.4034,
        "criteria_match": 0.0721,
        "combined": 0.2927,
    },
    "categories": {
        "gtfs_terminology": {"combined_avg": 0.3512, "count": 25},
        "gtfs_validation": {"combined_avg": 0.3137, "count": 18},
        "journey_planning": {"combined_avg": 0.2426, "count": 10},
        "route_analysis": {"combined_avg": 0.2901, "count": 75},
        "schedule_reasoning": {"combined_avg": 0.2525, "count": 38},
        "transit_operations": {"combined_avg": 0.3065, "count": 27},
    },
}

base_dict = {
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct (base)",
    "total_questions": 193,
    "overall": {
        "rouge_l": 0.129,
        "keyword_match": 0.3678,
        "criteria_match": 0.0203,
        "combined": 0.168,
    },
    "categories": {
        "gtfs_terminology": {"combined_avg": 0.342, "count": 25},
        "gtfs_validation": {"combined_avg": 0.2667, "count": 18},
        "journey_planning": {"combined_avg": 0.2967, "count": 10},
        "route_analysis": {"combined_avg": 0.0843, "count": 75},
        "schedule_reasoning": {"combined_avg": 0.1205, "count": 38},
        "transit_operations": {"combined_avg": 0.1927, "count": 27},
    },
}

## 10. Three-Way Comparison

In [ ]:
print("=" * 85)
print("  THREE-WAY COMPARISON: Base vs v0.1 vs v1.0")
print("=" * 85)

print(f"\n  {'Metric':<20} {'Base':>12} {'v0.1':>12} {'v1.0':>12} {'v0.1→v1.0':>12}")
print("  " + "-" * 70)

for metric in ["rouge_l", "keyword_match", "criteria_match", "combined"]:
    b = base_dict["overall"][metric]
    v01 = v01_dict["overall"][metric]
    v1 = v1_dict["overall"][metric]
    delta = v1 - v01
    sign = "+" if delta > 0 else ""
    print(f"  {metric:<20} {b:>12.4f} {v01:>12.4f} {v1:>12.4f} {sign}{delta:>11.4f}")

print(f"\n  {'Category':<20} {'Base':>12} {'v0.1':>12} {'v1.0':>12} {'v0.1→v1.0':>12}")
print("  " + "-" * 70)

all_cats = sorted(set(list(base_dict["categories"].keys()) + list(v1_dict["categories"].keys())))
for cat in all_cats:
    b = base_dict["categories"].get(cat, {}).get("combined_avg", 0)
    v01 = v01_dict["categories"].get(cat, {}).get("combined_avg", 0)
    v1 = v1_dict["categories"].get(cat, {}).get("combined_avg", 0)
    delta = v1 - v01
    sign = "+" if delta > 0 else ""
    print(f"  {cat:<20} {b:>12.4f} {v01:>12.4f} {v1:>12.4f} {sign}{delta:>11.4f}")

print("=" * 85)
overall_delta = v1_dict["overall"]["combined"] - v01_dict["overall"]["combined"]
print(f"\n  v0.1→v1.0 overall improvement: {overall_delta:+.4f}")
print(f"  v1.0 vs base improvement: {v1_dict['overall']['combined'] - base_dict['overall']['combined']:+.4f}")

## 11. Save Results

In [ ]:
from datetime import datetime, timezone

# Save v1.0 results
v1_output = {
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
    "report": v1_dict,
    "results": v1_results,
}
with open("umartransit_v1_results.json", "w") as f:
    json.dump(v1_output, f, indent=2, ensure_ascii=False)

# Save three-way comparison
comparison = {
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
    "questions_evaluated": len(questions),
    "base_model": base_dict,
    "umartransit_v01": v01_dict,
    "umartransit_v1": v1_dict,
}
with open("v1_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

print("Saved:")
print("  1. umartransit_v1_results.json — full v1.0 predictions")
print("  2. v1_comparison.json — three-way comparison")
print("\nDownload both from the Colab file browser (left panel).")